# Public reproduction copy
Source: researcher-supplied executed article notebook. Participant-level outputs and figures have been removed; only selected aggregate results remain. Input files and participant predictions are generated locally when you run the code and must not be uploaded automatically. Retained aggregate outputs are a transcription of the supplied execution, not a new run. The underlying 74-cell notebook is unchanged apart from output/metadata sanitization and this notice.


# SpaceSleepTwin — Notebook do artigo

Reproduz **todas** as tabelas do artigo *Predição prospectiva da latência do sono com vestíveis e diários* a partir dos dados públicos, na ordem em que as análises foram feitas:

| Parte | Conteúdo | Status no artigo |
|---|---|---|
| A | Manchester: desenvolvimento (v3) e reservado | desenvolvimento exploratório; reservado avaliado uma vez com modelos fixados |
| B | MeLiDos: replicação com plano fixado | confirmatório para as comparações principais |
| C | Cruzamento Manchester × MeLiDos | exploratório, feito após abrir os reservados |
| D | Conferência: cada número do artigo comparado com o recalculado | — |

Os reservados já foram abertos; reexecutar este notebook apenas reproduz a avaliação. Os dados são baixados por commit fixo e verificados por SHA-256 para pastas locais (`data_manchester/`, `data_melidos/`) que **não devem ir para o repositório público**, assim como os CSVs harmonizados e as previsões por participante.

Dependências: `numpy pandas scikit-learn lightgbm pyreadr matplotlib`. Tempo de execução: cerca de 10 minutos.

# Parte A — Manchester

## SpaceSleepTwin v3 — Previsão da latência do sono no estilo "competição" (Manchester)

**Objetivo.** Reduzir o erro absoluto médio (MAE) da latência do sono relatada no diário, usando o método de trabalho de competições de ML: validação que imita o uso real, otimização direta da métrica, engenharia de variáveis, análise de erro, pós-processamento e combinação — com um **conjunto final reservado, avaliado uma única vez**.

**Dados.** Didikoglu, Lucas & Brown (2023), *PNAS*, doi:10.1073/pnas.2301608120. Tabelas publicadas em `altugdidikoglu/light-sleep-inreallife`, fixadas no commit `ef57033`, com verificação SHA-256 (as mesmas da auditoria do repositório). Licença: confirmar os termos da versão exata antes de redistribuir. **Os arquivos são baixados para uma pasta local e não devem ser enviados ao repositório público.**

**Momento da previsão.** A pessoa está indo para a cama. Só pode entrar no modelo o que se sabe nesse momento:

| Trilha | Variáveis | Uso |
|---|---|---|
| **A — prospectiva** | horário de deitar, histórico pessoal de noites anteriores, luz diurna (`TAT250`, `time1000last`), calendário, perfil basal | previsão honesta |
| **B — concomitante** | A + luz nas 3 h antes do início do sono (`presleep3hr`) + horário da última luz > 1 lux | **não prospectiva**: `presleep3hr` é ancorada no início do sono medido pelo Fitbit na própria noite, e a última luz pode ocorrer depois de deitar. Serve para explorar associação (H1), não como previsor real |

**Versão 3** acrescenta: variáveis circadianas explícitas (horário de deitar relativo ao cronotipo, deitar antes do habitual, interação com horas acordada), discordância diário–Fitbit da noite anterior e uma **trilha de classificação** para P(latência > 30 min).

**Versão 2** acrescentou: pressão homeostática do sono (Processo S), perdas robustas (Huber e log), modelo em duas partes para noites longas e combinação com peso escolhido por validação interna (aninhada).

Limitações principais: diário autorrelatado com valores arredondados; cerca de 5 noites por pessoa; luz de sensor vestível; sem DLMO; nada disso valida o simulador nem se aplica diretamente a astronautas.

In [ ]:
import hashlib, json, pathlib, urllib.request, warnings
import numpy as np, pandas as pd, pyreadr
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import QuantileRegressor, LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
import lightgbm as lgb
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

SEED = 20261011
N_FOLDS = 5
N_BOOT = 2000
EVALUATE_HOLDOUT = True    # aberto uma única vez em 2026-10-11, com FINAL_MODELS e FINAL_CLF fixados na v3
DATA_DIR = pathlib.Path("data_manchester")   # fora do repositório público
OUT_DIR = pathlib.Path("results_manchester_competition")
OUT_DIR.mkdir(exist_ok=True)

## 1. Download fixado e verificação de integridade

In [ ]:
COMMIT = "ef570332cd6bd04cafea40a66f5a70e6f2206505"
SHA256 = {
    "light_daily_data": "d7bce0f4425067880432fe4acdbb031363890a1bfe19da8e90dc0ad6db8cd9c4",
    "sleep_diary":      "449085c3117b5b1ed9cc56b7f2dc02d8191dd898fa598952e93424fb46194176",
    "baseline_data":    "cf485c46e9db456693f7fb3ddccd1ac54315d886009ea318aae10395f800ef5d",
    "sleep_fitbit":     "d6ca44f158ef162e13394bcbc16246523bb190037fa4a262f256e4209a8bd095",
}
def load(name):
    DATA_DIR.mkdir(exist_ok=True)
    path = DATA_DIR / f"{name}.RData"
    if not path.exists():
        url = f"https://raw.githubusercontent.com/altugdidikoglu/light-sleep-inreallife/{COMMIT}/data/{name}.RData"
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == SHA256[name], f"SHA-256 diferente para {name}"
    return next(iter(pyreadr.read_r(str(path)).values()))

light, diary, base, fitbit = (load(n) for n in SHA256)
print({k: v.shape for k, v in zip(SHA256, (light, diary, base, fitbit))})

## 2. Junção, alvo e definição do conjunto reservado

A junção reproduz a do artigo (`Id` + `yday4`, um para um): luz do dia *d* com a noite que começa ao fim desse dia. Alvo: `hoursToSleep × 60` (minutos entre deitar e dormir, diário).

**Conjunto reservado:** a **última noite** de cada pessoa com pelo menos 3 noites. Ele imita o uso real (prever a próxima noite de uma pessoa conhecida). Os desfechos reservados são separados **antes** de qualquer exploração e o hash das chaves é registrado.

In [ ]:
df = light.merge(diary, on=["Id", "yday4"], validate="one_to_one")
df["lat_min"] = df["hoursToSleep"] * 60
df["evening"] = (df["datetimeGobed"] - pd.Timedelta(hours=12)).dt.normalize()
df = df.sort_values(["Id", "datetimeGobed"]).reset_index(drop=True)
df["night_idx"] = df.groupby("Id").cumcount()
df["n_nights"] = df.groupby("Id")["Id"].transform("size")
df["is_holdout"] = (df["n_nights"] >= 3) & (df["night_idx"] == df["n_nights"] - 1)

hold_keys = sorted(df.loc[df.is_holdout, "Id"] + "|" + df.loc[df.is_holdout, "evening"].dt.strftime("%Y-%m-%d"))
HOLDOUT_SHA = hashlib.sha256("\n".join(hold_keys).encode()).hexdigest()
print(f"Noites: {len(df)} | pessoas: {df.Id.nunique()}")
print(f"Reservadas: {df.is_holdout.sum()} noites de {df.loc[df.is_holdout,'Id'].nunique()} pessoas")
print("SHA-256 das chaves reservadas:", HOLDOUT_SHA)

## 3. Engenharia de variáveis (sem vazamento)

Todas as estatísticas pessoais usam **apenas noites anteriores** da mesma pessoa (`shift(1)` + janela expansiva). Como a noite reservada é sempre a última, seu desfecho nunca entra nas variáveis de outra noite.

In [ ]:
def past(s, fn):
    return s.shift(1).expanding().agg(fn)

g = df.groupby("Id", group_keys=False)
df["h_n"]        = df["night_idx"]
df["h_lat_med"]  = g["lat_min"].apply(lambda s: past(s, "median"))
df["h_lat_mean"] = g["lat_min"].apply(lambda s: past(s, "mean"))
df["h_lat_last"] = g["lat_min"].shift(1)
df["h_gobed_med"]= g["gobedTime"].apply(lambda s: past(s, "median"))
df["gobed_dev"]  = df["gobedTime"] - df["h_gobed_med"]
df["log_tat"]    = np.log1p(df["TAT250"])
df["h_tat_med"]  = g["log_tat"].apply(lambda s: past(s, "median"))
df["tat_dev"]    = df["log_tat"] - df["h_tat_med"]
df["dow"]        = df["evening"].dt.dayofweek
df["weekend_eve"]= df["dow"].isin([4, 5]).astype(int)       # sexta e sábado à noite
# trilha B (não prospectiva)
df["log_presleep"]        = np.log1p(df["presleep3hr"])
df["lightsoff_after_bed"] = df["time1last"] - (df["gobedTime"] + 24)

b = base[["Id", "age", "sex", "MSFsc", "psqi", "caffeine_base"]].copy()
b["age"]  = pd.to_numeric(b["age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
b["male"] = (b["sex"] == "Male").astype(float)
df = df.merge(b.drop(columns="sex"), on="Id", how="left")

# Processo S: pressão homeostática, conhecida ao deitar (vem da noite ANTERIOR)
prev = diary[["Id", "yday4", "datetimeWake", "sleepDuration", "sleepEfficiency"]].copy()
prev["yday4"] += 1
prev.columns = ["Id", "yday4", "prev_wake", "prev_dur", "prev_eff"]
df = df.merge(prev, on=["Id", "yday4"], how="left", validate="one_to_one")
df["hours_awake"] = (df["datetimeGobed"] - df["prev_wake"]).dt.total_seconds() / 3600
fb = fitbit[["Id", "yday4", "sleepDuration", "deepPerc", "wakeMinute"]].copy()
fb["yday4"] += 1
fb.columns = ["Id", "yday4", "prev_fb_dur", "prev_fb_deep", "prev_fb_wake"]
df = df.merge(fb, on=["Id", "yday4"], how="left", validate="one_to_one")
# Discordância diário–Fitbit da NOITE ANTERIOR (latência do diário − início detectado pelo Fitbit após deitar)
nd = diary[["Id", "yday4", "datetimeGobed", "hoursToSleep"]].merge(
    fitbit[["Id", "yday4", "startTime"]], on=["Id", "yday4"], how="left", validate="one_to_one")
nd["fb_lat"] = ((nd["startTime"] - nd["datetimeGobed"]).dt.total_seconds() / 60).clip(0, 240)
nd["prev_discord"] = (nd["hoursToSleep"] * 60 - nd["fb_lat"]).clip(-120, 120)
nd = nd[["Id", "yday4", "prev_discord"]].assign(yday4=lambda d: d["yday4"] + 1)
df = df.merge(nd, on=["Id", "yday4"], how="left", validate="one_to_one")
print(f"Cobertura da discordância da noite anterior: {df.prev_discord.notna().mean():.0%}")
print(f"Cobertura da noite anterior: diário {df.prev_dur.notna().mean():.0%}, Fitbit {df.prev_fb_dur.notna().mean():.0%}")

FEAT_A = ["gobedTime", "gobed_dev", "h_n", "h_lat_med", "h_lat_mean", "h_lat_last",
          "log_tat", "tat_dev", "time1000last", "dow", "weekend_eve",
          "age", "male", "MSFsc", "psqi", "caffeine_base"]
# Variáveis circadianas explícitas (definidas depois do merge do perfil basal, abaixo)
FEAT_S = FEAT_A + ["hours_awake", "prev_dur", "prev_eff", "prev_fb_dur", "prev_fb_deep", "prev_fb_wake"]
FEAT_B = FEAT_S + ["log_presleep", "lightsoff_after_bed"]

# Processo C aproximado pelo cronotipo (MSFsc = meio do sono em dias livres, corrigido)
df["bed_rel_chrono"] = df["gobedTime"] - df["MSFsc"]            # horas; mais negativo = deita mais cedo para seu cronotipo
df["early_vs_habit"] = (-df["gobed_dev"]).clip(lower=0)         # deitar antes do habitual: aproximação da zona de manutenção da vigília
df["awake_x_chrono"] = (df["hours_awake"] - 16) * (df["bed_rel_chrono"] + 7)
FEAT_C = FEAT_S + ["bed_rel_chrono", "early_vs_habit", "awake_x_chrono", "prev_discord"]

hold = df[df.is_holdout].copy()
HOLD_Y = hold.pop("lat_min")          # guardado; não usado até o fim
dev = df[~df.is_holdout].reset_index(drop=True)
print(f"Desenvolvimento: {len(dev)} noites, {dev.Id.nunique()} pessoas")

## 4. Exploração do alvo (somente desenvolvimento)

In [ ]:
y = dev["lat_min"]
print(y.describe().round(2))
print("\nValores mais frequentes (min):")
print(y.round(1).value_counts().head(10))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(y, bins=40); ax[0].set_title("Latência (min) — desenvolvimento"); ax[0].set_xlabel("min")
ax[1].hist(np.log1p(y), bins=30); ax[1].set_title("log(1 + latência)")
plt.tight_layout(); plt.show()

## 5. Validação cruzada

**Esquema:** `GroupKFold` por pessoa (5 dobras, sorteio fixo). Cada pessoa é prevista por modelos treinados **só com outras pessoas**; as variáveis dela vêm apenas de suas noites anteriores. Assim o modelo global nunca vê o futuro da pessoa avaliada.

**Avaliação principal:** noites com ao menos uma noite anterior (onde o histórico pessoal existe). A primeira noite de cada pessoa é reportada à parte.

**Modelos** (hiperparâmetros fixos, sem ajuste). `A` = variáveis prospectivas básicas; `S` = A + Processo S.

| Modelo | O que testa |
|---|---|
| `pop_median` | mediana da população de treino |
| `history` | **referência a ser batida**: mediana das noites anteriores da pessoa |
| `quantile_A`, `quantile_S` | regressão quantílica linear na mediana |
| `lgbm_A`, `lgbm_S` | LightGBM, perda L1 |
| `lgbm_S_huber` | perda de Huber (δ = 15 min) |
| `lgbm_S_log` | perda L1 sobre log(1 + latência) |
| `twopart_S` | duas partes: P(latência > 30 min) por regressão logística × distribuição de cada regime; a previsão é a **mediana da mistura** |
| `blend_nested` | histórico + `quantile_S`, com peso escolhido por validação **interna** dentro de cada dobra (sem olhar a dobra avaliada) |
| `quantile_C`, `lgbm_C` | S + variáveis circadianas e discordância diário–Fitbit (versão 3) |
| `lgbm_B` | trilha B, não prospectiva |

In [ ]:
rng = np.random.default_rng(SEED)
ids = np.array(sorted(dev.Id.unique()))
fold_of = dict(zip(ids, rng.permutation(np.arange(len(ids)) % N_FOLDS)))
dev["fold"] = dev.Id.map(fold_of)

LGB_PARAMS = dict(objective="l1", n_estimators=300, learning_rate=0.03, num_leaves=7,
                  min_child_samples=15, subsample=0.8, subsample_freq=1,
                  colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)

def fit_lgb(X, y, seeds=5):
    return [lgb.LGBMRegressor(**LGB_PARAMS, random_state=SEED + s).fit(X, y) for s in range(seeds)]
def pred_lgb(models, X):
    return np.mean([m.predict(X) for m in models], axis=0)
def fit_quantile(X, y):
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         QuantileRegressor(quantile=0.5, alpha=0.01, solver="highs")).fit(X, y)

LONG_THR = 30
GRID = np.arange(0, 180.5, 0.5)
def fit_twopart(tr, feats):
    long = (tr["lat_min"] > LONG_THR).to_numpy()
    clf = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                        LogisticRegression(C=0.3, max_iter=5000)).fit(tr[feats], long)
    short = tr[~long]
    reg = fit_quantile(short[feats], short["lat_min"])
    res_short = np.sort(short["lat_min"].to_numpy() - reg.predict(short[feats]))
    long_vals = np.sort(tr.loc[long, "lat_min"].to_numpy())
    return clf, reg, res_short, long_vals
def pred_twopart(model, te, feats):
    clf, reg, res_short, long_vals = model
    p = clf.predict_proba(te[feats])[:, 1]; ms = reg.predict(te[feats])
    Fl = np.searchsorted(long_vals, GRID, side="right") / len(long_vals)
    out = []
    for pi, mi in zip(p, ms):
        Fs = np.searchsorted(res_short, GRID - mi, side="right") / len(res_short)
        out.append(GRID[np.argmax((1 - pi) * Fs + pi * Fl >= 0.5)])
    return np.array(out)

def hist_pred(tr, te):
    pop = tr["lat_min"].median()
    return te["h_lat_med"].fillna(pop).to_numpy()

W_GRID = np.round(np.arange(0, 1.01, 0.1), 1)
def choose_blend_weight(tr, n_inner=4, seed=SEED):
    # peso de quantile_S escolhido só com o treino, por validação interna agrupada por pessoa
    r = np.random.default_rng(seed)
    ids_in = np.array(sorted(tr.Id.unique()))
    inner = dict(zip(ids_in, r.permutation(np.arange(len(ids_in)) % n_inner)))
    fi = tr.Id.map(inner).to_numpy()
    h = np.full(len(tr), np.nan); q = np.full(len(tr), np.nan)
    for k in range(n_inner):
        a, b = tr[fi != k], tr[fi == k]
        h[fi == k] = hist_pred(a, b)
        q[fi == k] = fit_quantile(a[FEAT_S], a["lat_min"]).predict(b[FEAT_S])
    ok = (tr["h_n"] >= 1).to_numpy(); yy = tr["lat_min"].to_numpy()
    errs = [np.mean(np.abs(yy[ok] - ((1 - w) * h[ok] + w * q[ok]))) for w in W_GRID]
    return float(W_GRID[int(np.argmin(errs))])

def predict_block(tr, te):
    out, models = {}, {}
    out["pop_median"] = np.full(len(te), tr["lat_min"].median())
    out["history"] = hist_pred(tr, te)
    out["quantile_A"] = fit_quantile(tr[FEAT_A], tr["lat_min"]).predict(te[FEAT_A])
    out["quantile_S"] = fit_quantile(tr[FEAT_S], tr["lat_min"]).predict(te[FEAT_S])
    out["quantile_C"] = fit_quantile(tr[FEAT_C], tr["lat_min"]).predict(te[FEAT_C])
    for name, feats, kw, logt in [("lgbm_A", FEAT_A, {}, False), ("lgbm_S", FEAT_S, {}, False),
                                  ("lgbm_S_huber", FEAT_S, dict(objective="huber", alpha=15.0), False),
                                  ("lgbm_S_log", FEAT_S, {}, True), ("lgbm_C", FEAT_C, {}, False),
                                  ("lgbm_B", FEAT_B, {}, False)]:
        yt = np.log1p(tr["lat_min"]) if logt else tr["lat_min"]
        ms = [lgb.LGBMRegressor(**{**LGB_PARAMS, **kw}, random_state=SEED + s).fit(tr[feats], yt) for s in range(5)]
        p = pred_lgb(ms, te[feats])
        out[name] = np.expm1(p) if logt else p
        models[name] = ms
    out["twopart_S"] = pred_twopart(fit_twopart(tr, FEAT_S), te, FEAT_S)
    w = choose_blend_weight(tr)
    out["blend_nested"] = (1 - w) * out["history"] + w * out["quantile_S"]
    models["blend_weight"] = w
    return out, models

def mae(e): return float(np.mean(np.abs(e)))

oof = {}; perm_rows = []; fold_weights = []
prng = np.random.default_rng(SEED + 1)
for f in range(N_FOLDS):
    tr, te = dev[dev.fold != f], dev[dev.fold == f]
    preds, models = predict_block(tr, te)
    fold_weights.append(models["blend_weight"])
    for k, v in preds.items():
        oof.setdefault(k, pd.Series(index=dev.index, dtype=float)).loc[te.index] = v
    # importância por permutação, medida na dobra fora do treino
    for name, feats in [("lgbm_C", FEAT_C), ("lgbm_B", FEAT_B)]:
        base_err = mae(te["lat_min"] - pred_lgb(models[name], te[feats]))
        for feat in feats:
            Xp = te[feats].copy(); Xp[feat] = prng.permutation(Xp[feat].to_numpy())
            perm_rows.append(dict(model=name, feature=feat, fold=f,
                                  delta_mae=mae(te["lat_min"] - pred_lgb(models[name], Xp)) - base_err))
oof = pd.DataFrame(oof)
print("OOF pronto:", oof.shape)
print("Peso de quantile_S escolhido em cada dobra (validação interna):", fold_weights)

## 6. Resultados fora da amostra

In [ ]:
MAIN = dev["h_n"] >= 1
def summarize(mask, cols):
    rows = []
    for c in cols:
        e = dev.loc[mask, "lat_min"] - oof.loc[mask, c]
        per = e.abs().groupby(dev.loc[mask, "Id"]).mean()
        rows.append(dict(model=c, n=int(mask.sum()), MAE=mae(e), MedAE=float(e.abs().median()),
                         MAE_por_pessoa=float(per.mean()), RMSE=float(np.sqrt(np.mean(e**2)))))
    return pd.DataFrame(rows).set_index("model").round(2)

MODELS = list(oof.columns)
print("Principal — noites com histórico pessoal")
res_main = summarize(MAIN, MODELS); display(res_main)
print("Primeira noite de cada pessoa (sem histórico)")
display(summarize(~MAIN, MODELS))

### Bootstrap pareado por pessoa (diferença de MAE contra o histórico)

Reamostra pessoas, mantendo fixas as previsões fora da amostra. Valores negativos = melhor que o histórico. Intervalos descritivos: não contemplam o reajuste dos modelos nem as várias comparações feitas neste notebook.

In [ ]:
def paired_boot(model, ref="history", mask=MAIN, n=N_BOOT, seed=SEED):
    d = dev.loc[mask]
    diff = (d["lat_min"] - oof.loc[mask, model]).abs() - (d["lat_min"] - oof.loc[mask, ref]).abs()
    by = diff.groupby(d["Id"]).agg(["sum", "count"])
    s, c = by["sum"].to_numpy(), by["count"].to_numpy()
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(s), size=(n, len(s)))
    boots = s[idx].sum(1) / c[idx].sum(1)
    per_person = diff.groupby(d["Id"]).mean()
    return dict(model=model, delta_MAE=round(diff.mean(), 3),
                IC95_inf=round(np.percentile(boots, 2.5), 3), IC95_sup=round(np.percentile(boots, 97.5), 3),
                melhorou=int((per_person < -1e-9).sum()), piorou=int((per_person > 1e-9).sum()))

boot = pd.DataFrame([paired_boot(m) for m in MODELS if m != "history"]).set_index("model")
display(boot)

## 7. Análise de erro

Onde o erro está concentrado? Em competições, é aqui que surgem as próximas variáveis.

In [ ]:
CANDIDATES = ["quantile_A", "quantile_S", "quantile_C", "lgbm_C", "lgbm_A", "lgbm_S", "lgbm_S_huber", "lgbm_S_log", "twopart_S", "blend_nested"]
best = res_main.loc[CANDIDATES].MAE.idxmin()
d = dev.loc[MAIN].copy()
d["err_hist"] = (d.lat_min - oof.loc[MAIN, "history"]).abs()
d["err_best"] = (d.lat_min - oof.loc[MAIN, best]).abs()
d["faixa"] = pd.cut(d.lat_min, [-0.1, 5, 15, 30, 600], labels=["0–5", "5–15", "15–30", ">30"])
tab = d.groupby("faixa", observed=True).agg(noites=("lat_min", "size"), MAE_hist=("err_hist", "mean"),
                                             MAE_best=("err_best", "mean"), soma_erro_hist=("err_hist", "sum"))
tab["% do erro total (hist)"] = 100 * tab.soma_erro_hist / tab.soma_erro_hist.sum()
print("Melhor candidato prospectivo na validação:", best)
display(tab.drop(columns="soma_erro_hist").round(2))

top = d.err_hist.sort_values(ascending=False)
print(f"As 10% piores noites concentram {100*top.head(int(len(top)*0.1)).sum()/top.sum():.1f}% do erro absoluto do histórico.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(d.lat_min, d.lat_min - oof.loc[MAIN, best], s=12, alpha=.6)
ax[0].axhline(0, c="k", lw=.8); ax[0].set_xlabel("latência observada (min)"); ax[0].set_ylabel("resíduo")
ax[0].set_title(f"Resíduos — {best}")
pp = d.groupby("Id")[["err_hist", "err_best"]].mean()
lim = pp.max().max() * 1.05
ax[1].scatter(pp.err_hist, pp.err_best, s=14, alpha=.7); ax[1].plot([0, lim], [0, lim], "k--", lw=.8)
ax[1].set_xlabel("MAE por pessoa — histórico"); ax[1].set_ylabel(f"MAE por pessoa — {best}")
ax[1].set_title("Abaixo da diagonal = melhorou")
plt.tight_layout(); plt.show()

### Importância por permutação (fora da amostra, média das dobras)

In [ ]:
perm = pd.DataFrame(perm_rows).groupby(["model", "feature"]).delta_mae.mean().unstack(0).round(3)
display(perm.sort_values("lgbm_C", ascending=False))
print("Aumento do MAE ao embaralhar a variável; valores perto de zero ou negativos = sem contribuição.")

## 8. Pós-processamento e combinação

- **Arredondamento para 5 min**, porque o diário concentra valores em múltiplos de 5.
- A combinação honesta já está em `blend_nested` (peso escolhido sem olhar a dobra avaliada). A curva abaixo, com peso escolhido no próprio OOF, é **otimista** e serve só para comparar com ela.

In [ ]:
yv = dev.loc[MAIN, "lat_min"]
snap = lambda p: np.round(p / 5) * 5
pp_rows = [dict(model=m, MAE=mae(yv - oof.loc[MAIN, m]), MAE_arred_5min=mae(yv - snap(oof.loc[MAIN, m])))
           for m in ["history"] + CANDIDATES]
display(pd.DataFrame(pp_rows).set_index("model").round(2))

ws = np.round(np.arange(0, 1.01, 0.1), 1)
blend = [mae(yv - ((1 - w) * oof.loc[MAIN, "history"] + w * oof.loc[MAIN, "quantile_S"])) for w in ws]
W_BEST = float(ws[int(np.argmin(blend))])
print(f"Otimista (peso escolhido no OOF): w={W_BEST}  ->  MAE {min(blend):.2f}")
print(f"Honesto (blend_nested):            MAE {res_main.loc['blend_nested','MAE']:.2f}")
plt.plot(ws, blend, marker="o"); plt.axhline(res_main.loc["blend_nested","MAE"], ls="--", c="gray")
plt.xlabel("peso de quantile_S"); plt.ylabel("MAE OOF"); plt.show()

## 9. Trilha de classificação: P(latência > 30 min)

Latência > 30 min é um critério clínico comum de dificuldade para iniciar o sono. Aqui a pergunta muda: **a noite será difícil?**

**Definido antes de rodar.** Desfecho principal: `lat_min > 30`. Sensibilidade: `≥ 30` (o diário concentra muitas respostas em exatamente 30 min, então o limiar muda bastante o número de eventos). Mesmas dobras por pessoa da regressão; avaliação nas noites com histórico.

| Modelo | O que é |
|---|---|
| `base_rate` | prevalência no treino |
| `hist_rate` | **referência**: proporção de noites longas no histórico da pessoa, suavizada para a prevalência (prior Beta com peso 2) |
| `logit_S` | logística regularizada (C = 0,3) com variáveis S |
| `logit_C` | logística regularizada com variáveis C |
| `logit_C_hist` | logística C + `hist_rate` como variável |
| `lgbm_clf_C` | LightGBM binário pequeno, variáveis C + `hist_rate` |

Métricas: Brier e log-loss (calibração + discriminação; menor é melhor), AUC e PR-AUC (discriminação). Com poucos eventos por dobra, os intervalos serão largos; um modelo simples tem mais chance que um complexo.

In [ ]:
PRIOR_W = 2.0
def personal_rate(frame, y_event, p0):
    # proporção de noites longas ANTERIORES da pessoa, suavizada para p0
    ev = y_event.astype(float)
    k = ev.groupby(frame["Id"]).transform(lambda s: s.shift(1).expanding().sum()).fillna(0)
    n = frame["h_n"]
    return ((k + PRIOR_W * p0) / (n + PRIOR_W)).to_numpy()

def logit():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         LogisticRegression(C=0.3, max_iter=5000))
CLF_LGB = dict(objective="binary", n_estimators=200, learning_rate=0.03, num_leaves=4,
               min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
               reg_lambda=5.0, verbose=-1)

def clf_block(tr, te, ytr, k_te_event_hist):
    p0 = ytr.mean()
    out = {"base_rate": np.full(len(te), p0)}
    hr_tr = personal_rate(tr, ytr, p0)
    hr_te = k_te_event_hist(p0)
    out["hist_rate"] = hr_te
    out["logit_S"] = logit().fit(tr[FEAT_S], ytr).predict_proba(te[FEAT_S])[:, 1]
    out["logit_C"] = logit().fit(tr[FEAT_C], ytr).predict_proba(te[FEAT_C])[:, 1]
    Xtr = tr[FEAT_C].assign(hist_rate=hr_tr); Xte = te[FEAT_C].assign(hist_rate=hr_te)
    out["logit_C_hist"] = logit().fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    ps = [lgb.LGBMClassifier(**CLF_LGB, random_state=SEED + s).fit(Xtr, ytr).predict_proba(Xte)[:, 1] for s in range(5)]
    out["lgbm_clf_C"] = np.mean(ps, axis=0)
    return out

def run_clf(threshold_fn, label):
    yev = threshold_fn(dev["lat_min"]).astype(int)
    oofc = {}
    for f in range(N_FOLDS):
        tr, te = dev[dev.fold != f], dev[dev.fold == f]
        # histórico da pessoa avaliada vem das próprias noites anteriores (já no conjunto de teste, ordenado)
        hist_fn = lambda p0, te=te: personal_rate(te, yev.loc[te.index], p0)
        preds = clf_block(tr, te, yev.loc[tr.index], hist_fn)
        for k, v in preds.items():
            oofc.setdefault(k, pd.Series(index=dev.index, dtype=float)).loc[te.index] = v
    oofc = pd.DataFrame(oofc)
    m = MAIN
    yy = yev[m]
    rows = []
    for c in oofc.columns:
        p = oofc.loc[m, c].clip(1e-4, 1 - 1e-4)
        rows.append(dict(model=c, eventos=int(yy.sum()), n=int(m.sum()), Brier=brier_score_loss(yy, p),
                         LogLoss=log_loss(yy, p), AUC=roc_auc_score(yy, p) if c != "base_rate" else np.nan,
                         PR_AUC=average_precision_score(yy, p)))
    res = pd.DataFrame(rows).set_index("model").round(4)
    # bootstrap pareado por pessoa da diferença de Brier contra hist_rate
    d = dev.loc[m]
    bs = []
    r = np.random.default_rng(SEED)
    for c in oofc.columns:
        if c == "hist_rate": continue
        diff = (oofc.loc[m, c] - yy) ** 2 - (oofc.loc[m, "hist_rate"] - yy) ** 2
        by = diff.groupby(d["Id"]).agg(["sum", "count"]); s_, n_ = by["sum"].to_numpy(), by["count"].to_numpy()
        idx = r.integers(0, len(s_), size=(N_BOOT, len(s_)))
        boots = s_[idx].sum(1) / n_[idx].sum(1)
        bs.append(dict(model=c, delta_Brier=round(diff.mean(), 4),
                       IC95_inf=round(np.percentile(boots, 2.5), 4), IC95_sup=round(np.percentile(boots, 97.5), 4)))
    print(f"=== {label}: {int(yy.sum())} eventos em {int(m.sum())} noites ({yy.mean():.1%}) ===")
    display(res)
    print("Diferença de Brier contra hist_rate (negativo = melhor):")
    display(pd.DataFrame(bs).set_index("model"))
    return oofc, res, yev

oof_clf, res_clf, yev_main = run_clf(lambda y: y > 30, "Principal: latência > 30 min")
oof_clf_s, res_clf_s, _ = run_clf(lambda y: y >= 30, "Sensibilidade: latência ≥ 30 min")

### Calibração (principal) — previsto vs. observado por faixa de probabilidade

In [ ]:
best_clf = res_clf.drop(index=["base_rate"]).Brier.idxmin()
yy = yev_main[MAIN]
fig, ax = plt.subplots(figsize=(5, 4))
for c in ["hist_rate", best_clf] if best_clf != "hist_rate" else ["hist_rate"]:
    p = oof_clf.loc[MAIN, c]
    bins = pd.qcut(p.rank(method="first"), 4, labels=False)
    t = pd.DataFrame({"p": p, "y": yy, "b": bins}).groupby("b").mean()
    ax.plot(t.p, t.y, marker="o", label=c)
ax.plot([0, .6], [0, .6], "k--", lw=.8); ax.set_xlabel("probabilidade prevista"); ax.set_ylabel("proporção observada")
ax.set_title("Calibração por quartis"); ax.legend(); plt.show()
print("Melhor classificador por Brier:", best_clf)

## 10. Conjunto reservado — avaliar uma única vez

1. Revise as seções 6–8 e **escreva abaixo, antes de abrir**, os modelos finais (no máximo 3, sempre com o histórico como referência).
2. Faça commit do notebook com `FINAL_MODELS` preenchido e o hash da seção 2.
3. Só então mude `EVALUATE_HOLDOUT = True` e rode esta seção. Não volte a ajustar nada depois de ver o resultado.

A trilha B não entra aqui: ela não é prospectiva. O mesmo vale para a classificação: fixe `FINAL_CLF` antes de abrir.

In [ ]:
FINAL_MODELS = ["history", "quantile_S", "blend_nested"]   # proposta: revise e fixe ANTES de abrir

# reconstrói as variáveis do reservado com histórico vindo só das noites de desenvolvimento
preds_hold, mods_hold = predict_block(dev, hold.assign(lat_min=np.nan))
print("Peso da combinação para o reservado (validação interna no desenvolvimento):", mods_hold["blend_weight"])
hold_pred = pd.DataFrame({m: preds_hold[m] for m in FINAL_MODELS}, index=hold.index)
hold_pred.insert(0, "Id", hold.Id.values)
hold_pred.to_csv(OUT_DIR / "holdout_predictions_sealed.csv", index=False)
print("Previsões do reservado salvas (sem desfechos).", hold_pred.shape)

FINAL_CLF = ["hist_rate", "logit_C_hist"]   # proposta: revise e fixe ANTES de abrir
ytr_ev = (dev["lat_min"] > 30).astype(int)
hold_hist = lambda p0: ((hold.Id.map(ytr_ev.groupby(dev.Id).sum()).fillna(0) + PRIOR_W * p0) / (hold["h_n"] + PRIOR_W)).to_numpy()
clf_hold = clf_block(dev, hold, ytr_ev, hold_hist)
clf_hold_df = pd.DataFrame({c: clf_hold[c] for c in FINAL_CLF}, index=hold.index).assign(Id=hold.Id.values)
clf_hold_df.to_csv(OUT_DIR / "holdout_clf_predictions_sealed.csv", index=False)

if EVALUATE_HOLDOUT:
    yh = (HOLD_Y > 30).astype(int)
    display(pd.DataFrame([dict(model=c, eventos=int(yh.sum()), Brier=brier_score_loss(yh, clf_hold_df[c]),
                               AUC=roc_auc_score(yh, clf_hold_df[c])) for c in FINAL_CLF]).set_index("model").round(4))
    rows = []
    for m in FINAL_MODELS:
        e = HOLD_Y.values - hold_pred[m].values
        rows.append(dict(model=m, n=len(e), MAE=mae(e), MedAE=float(np.median(np.abs(e)))))
    display(pd.DataFrame(rows).set_index("model").round(2))
else:
    print("Reservado ainda lacrado. EVALUATE_HOLDOUT = False.")

              eventos   Brier     AUC
model                                
hist_rate          13  0.1302  0.7623
logit_C_hist       13  0.1432  0.6510

               n    MAE  MedAE
model                         
history       80  11.94   5.00
quantile_S    80  11.51   4.89
blend_nested  80  11.57   5.63

## 11. Exportação

In [ ]:
oof_out = dev[["Id", "evening", "h_n", "lat_min"]].join(oof)
oof_out.to_csv(OUT_DIR / "oof_predictions.csv", index=False)   # contém dados de participantes: não publicar
summary = dict(
    source=f"altugdidikoglu/light-sleep-inreallife@{COMMIT}", seed=SEED, folds=N_FOLDS,
    holdout_keys_sha256=HOLDOUT_SHA, holdout_evaluated=EVALUATE_HOLDOUT,
    n_dev=len(dev), n_main=int(MAIN.sum()), lgb_params=LGB_PARAMS,
    metrics_main=res_main.reset_index().to_dict("records"),
    paired_bootstrap_vs_history=boot.reset_index().to_dict("records"),
    classification_main=res_clf.reset_index().to_dict("records"),
    classification_sensitivity=res_clf_s.reset_index().to_dict("records"),
    blend_weights_nested_per_fold=fold_weights, blend_weight_optimistic_oof=W_BEST,
    notes=["Trilha B não é prospectiva.", "Bootstrap condicional, sem reajuste nem correção para múltiplas comparações.",
           "Arredondamento e peso otimista escolhidos no OOF; blend_nested usa validação interna."])
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
print("Salvo em", OUT_DIR.resolve())

### Incerteza no reservado (bootstrap por noite; uma noite por pessoa)

In [ ]:
hp = pd.read_csv(OUT_DIR/"holdout_predictions_sealed.csv"); y = HOLD_Y.values
r = np.random.default_rng(SEED); idx = r.integers(0, len(y), size=(N_BOOT, len(y)))
for m in ["quantile_S","blend_nested"]:
    d = np.abs(y-hp[m].values)-np.abs(y-hp["history"].values); b = d[idx].mean(1)
    print(m, "ΔMAE vs history", round(d.mean(),3), np.round(np.percentile(b,[2.5,97.5]),3), "melhor/pior", (d<0).sum(), (d>0).sum())
yh=(y>30).astype(int); p0=(dev.lat_min>30).mean()
print("Brier prevalência no reservado", round(brier_score_loss(yh, np.full(len(yh),p0)),4))
for c in ["hist_rate","logit_C_hist"]:
    d=(clf_hold_df[c].values-yh)**2-(p0-yh)**2; b=d[idx].mean(1)
    print(c,"ΔBrier vs prevalência",round(d.mean(),4),np.round(np.percentile(b,[2.5,97.5]),4))

In [ ]:
# Exportação harmonizada para o cruzamento entre coortes (arquivo local: contém dados de participantes, não publicar)
COMMON = ["gobedTime", "gobed_dev", "h_n", "h_lat_med", "h_lat_mean", "h_lat_last", "log_tat", "tat_dev",
          "time1000last", "dow", "weekend_eve", "age", "male", "MSFsc", "hours_awake", "prev_dur", "prev_eff"]
xp = df[["Id", "evening", "is_holdout", "lat_min"] + COMMON].copy()
xp["prev_eff"] = xp["prev_eff"] / 100          # Manchester registra eficiência em %
xp["cohort"] = "Manchester"; xp["site"] = "Manchester"
xp.to_csv(OUT_DIR / "harmonized_manchester.csv", index=False)
print("Exportado:", xp.shape)

In [ ]:
# Guarda os resultados de Manchester para a conferência final
ART = {}
yM = HOLD_Y.values
rM = np.random.default_rng(SEED); idxM = rM.integers(0, len(yM), size=(N_BOOT, len(yM)))
def _boot_rows(y, P, ref, models, idx):
    out = {}
    for m in models:
        d = np.abs(y - P[m]) - np.abs(y - P[ref]); b = d[idx].mean(1)
        out[m] = (float(d.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)), int((d < 0).sum()), int((d > 0).sum()))
    return out
PM = {m: preds_hold[m] for m in ["history", "quantile_S", "blend_nested"]}
yhM = (yM > 30).astype(int); p0M = (dev.lat_min > 30).mean()
ART["man"] = dict(
    dev=res_main.copy(), boot_dev=boot.copy(), clf_dev=res_clf.copy(), clf_sens=res_clf_s.copy(),
    hold_mae={m: float(np.mean(np.abs(yM - PM[m]))) for m in PM},
    hold_boot=_boot_rows(yM, PM, "history", ["quantile_S", "blend_nested"], idxM),
    hold_clf={c: (brier_score_loss(yhM, clf_hold_df[c]), roc_auc_score(yhM, clf_hold_df[c])) for c in FINAL_CLF},
    hold_base_brier=brier_score_loss(yhM, np.full(len(yhM), p0M)),
    hold_dbrier=(lambda d: (float(d.mean()), float(np.percentile(d[idxM].mean(1), 2.5)), float(np.percentile(d[idxM].mean(1), 97.5))))(
        (clf_hold_df["hist_rate"].values - yhM) ** 2 - (p0M - yhM) ** 2),
    n_hold=len(yM), ev_hold=int(yhM.sum()), n_main=int(MAIN.sum()), ev_dev=int((dev.loc[MAIN, "lat_min"] > 30).sum()),
    lat=(float(dev.lat_min.median()), float(dev.lat_min.mean()), float((dev.lat_min % 5 == 0).mean())),
)
print("Manchester guardado.")

# Parte B — MeLiDos

## SpaceSleepTwin — Replicação independente no MeLiDos (comparação com Manchester)

**Pergunta.** O padrão encontrado em Manchester se repete numa coorte independente? Lá, a mediana pessoal da latência (`history`) não foi superada de forma demonstrável por nenhum modelo, nem em minutos nem em risco de noite longa.

**Dados.** Estudo de campo MeLiDos (Guidolin et al., 2024, doi:10.1186/s12889-024-20206-4): diários de sono, cronotipo, dados demográficos e luz melanópica (mEDI) medida em óculos, por minuto. Cada local tem seu repositório em `github.com/MeLiDosProject`, todos CC BY 4.0, fixados por commit e verificados por SHA-256. **Os arquivos ficam numa pasta local e não devem ir para o repositório público.**

## Plano fixado antes de olhar os desfechos

1. **Locais:** BAUA, FUSPCEU, IZTECH, KNUST, MPI, RISE, THUAS, TUM. **UCR (Costa Rica) é excluído**: seus diários já foram analisados no teste lunar do SpaceSleepTwin, então não são independentes.
2. **Desfecho:** `sleepdelay` (min) do diário, contado a partir de `sleepprep` (início da tentativa de dormir). Em Manchester era de "deitar" a "dormir": o construto é próximo, não idêntico.
3. **Reservado:** última noite de cada pessoa com ≥ 3 noites, a mesma regra de Manchester.
4. **Modelos e hiperparâmetros:** os mesmos da v3, **sem nenhum ajuste**. Comparações principais definidas antes: regressão `history` × `quantile_S` × `blend_nested`; classificação (> 30 min) `hist_rate` × `logit_C_hist`.
5. **Variáveis:** equivalentes às de Manchester quando existem. Não existem aqui: PSQI, cafeína basal, Fitbit e discordância diário–Fitbit; são **removidas**, não imputadas.
6. **Secundárias, definidas antes:** (a) `quantile_S_light`: S + luz média nos 3 h **antes de deitar**, conhecida no momento da previsão (versão prospectiva de H1); (b) validação **deixando um local inteiro de fora**.
7. Como nada é escolhido a partir dos resultados, o reservado é avaliado nesta mesma execução. Tudo é reportado, seja qual for o resultado.

In [ ]:
import hashlib, json, pathlib, urllib.request, warnings
import numpy as np, pandas as pd, pyreadr
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import QuantileRegressor, LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
import lightgbm as lgb
warnings.filterwarnings("ignore"); pd.set_option("display.width", 170)
SEED, N_FOLDS, N_BOOT = 20261011, 5, 2000
DATA_DIR = pathlib.Path("data_melidos"); OUT_DIR = pathlib.Path("results_melidos_replication")
OUT_DIR.mkdir(exist_ok=True)

## 1. Download fixado por commit e integridade

In [ ]:
SITES = {  # local: (repositório, commit)
    "BAUA":    ("BroszioEtAl_Dataset_2025",        "a69ff3835c2ddef1379903ca6089eeb1a57c67c2"),
    "FUSPCEU": ("BaezaEtAl_Dataset_2025",          "2badb6e3302e898a5801cb35a3719bd0ec4b9aad"),
    "IZTECH":  ("DidikogluEtAl_Dataset_2025",      "ac7dda352be2ff3e30a777dd8772b45530beffa2"),
    "KNUST":   ("AkuffoEtAl_Dataset_2025",         "ea49a38edfb1f6397302f37776a4496ed0ca2dd5"),
    "MPI":     ("GuidolinEtAl_Dataset_2025",       "cd0099b0eed758692bdc9f23071d4a052ff8f72f"),
    "RISE":    ("NilssonTengelinEtAl_Dataset_2026","12ca7139b78949ab84e0be92822d03910c1e47c8"),
    "THUAS":   ("AertsEtAl_Dataset_2025",          "19bf83f353e1cbdd33cd22621b2484bc3ead0c7d"),
    "TUM":     ("HildenEtAl_Dataset_2025",         "618fda8521f3cf581661ceb2c026e3de7cd9ba82"),
}
FILES = {"sleepdiaries": "continuous/sleepdiaries", "chronotype": "chronotype",
         "demographics": "demographics", "light": "light/light_glasses_1minute"}
checksums = {}
def load(site, key):
    repo, commit = SITES[site]
    path = DATA_DIR / site / f"{key}.RData"; path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        url = f"https://raw.githubusercontent.com/MeLiDosProject/{repo}/{commit}/data/imported/{FILES[key]}.RData"
        path.write_bytes(urllib.request.urlopen(url, timeout=120).read())
    checksums[f"{site}/{key}"] = hashlib.sha256(path.read_bytes()).hexdigest()
    return next(iter(pyreadr.read_r(str(path)).values()))

raw = {s: {k: load(s, k) for k in FILES} for s in SITES}
(OUT_DIR / "source_checksums.json").write_text(json.dumps(
    {"commits": {s: v[1] for s, v in SITES.items()}, "sha256": checksums}, indent=2))
print({s: len(raw[s]["sleepdiaries"]) for s in SITES})

## 2. Montagem das noites, reservado e variáveis

Todas as estatísticas pessoais usam só noites anteriores. A luz vem dos óculos (mEDI por minuto); minutos marcados como lacuna implícita são tratados como ausentes, e janelas com menos de 50% de cobertura viram `NaN`.

Limitação: os participantes podem tirar os óculos à noite, então a luz "antes de deitar" pode registrar o ambiente onde os óculos foram deixados.

In [ ]:
def clock(ts):  # horas relativas à meia-noite, como em Manchester (22:30 -> -1.5)
    h = ts.dt.hour + ts.dt.minute / 60
    return np.where(h > 12, h - 24, h)

rows = []
for s in SITES:
    d = raw[s]["sleepdiaries"].copy()
    d["site"] = s
    rows.append(d)
df = pd.concat(rows, ignore_index=True)
df = df.dropna(subset=["bedtime", "sleepdelay"]).copy()
df["lat_min"] = df["sleepdelay"].astype(float)
df["evening"] = (df["bedtime"] - pd.Timedelta(hours=12)).dt.normalize()
df = df.sort_values(["Id", "bedtime"]).drop_duplicates(["Id", "evening"], keep="first").reset_index(drop=True)
df["night_idx"] = df.groupby("Id").cumcount()
df["n_nights"] = df.groupby("Id")["Id"].transform("size")
df["is_holdout"] = (df["n_nights"] >= 3) & (df["night_idx"] == df["n_nights"] - 1)
keys = sorted(df.loc[df.is_holdout, "Id"] + "|" + df.loc[df.is_holdout, "evening"].dt.strftime("%Y-%m-%d"))
HOLDOUT_SHA = hashlib.sha256("\n".join(keys).encode()).hexdigest()
print(f"Noites: {len(df)} | pessoas: {df.Id.nunique()} | locais: {df.site.nunique()}")
print(f"Reservadas: {df.is_holdout.sum()} | SHA-256 das chaves: {HOLDOUT_SHA}")

# noite anterior (Processo S)
g = df.groupby("Id")
df["prev_wake"] = g["wake"].shift(1)
df["prev_dur"] = g["sleep_duration"].shift(1) * 60
inbed = (df["out_ofbed"] - df["bedtime"]).dt.total_seconds() / 3600
df["prev_eff"] = (df["sleep_duration"] / inbed).groupby(df["Id"]).shift(1)
df["hours_awake"] = (df["bedtime"] - df["prev_wake"]).dt.total_seconds() / 3600
df.loc[(df.hours_awake <= 0) | (df.hours_awake > 30), ["hours_awake"]] = np.nan

def past(s, fn): return s.shift(1).expanding().agg(fn)
gg = df.groupby("Id", group_keys=False)
df["gobedTime"] = clock(df["bedtime"])
df["h_n"] = df["night_idx"]
df["h_lat_med"] = gg["lat_min"].apply(lambda s: past(s, "median"))
df["h_lat_mean"] = gg["lat_min"].apply(lambda s: past(s, "mean"))
df["h_lat_last"] = gg["lat_min"].shift(1)
df["h_gobed_med"] = gg["gobedTime"].apply(lambda s: past(s, "median"))
df["gobed_dev"] = df["gobedTime"] - df["h_gobed_med"]
df["dow"] = df["evening"].dt.dayofweek
df["weekend_eve"] = df["dow"].isin([4, 5]).astype(int)

# perfil basal
chrono = pd.concat([raw[s]["chronotype"][["Id", "msf_sc"]] for s in SITES])
demo = pd.concat([raw[s]["demographics"][["Id", "age", "sex"]] for s in SITES])
df = df.merge(chrono, on="Id", how="left").merge(demo, on="Id", how="left")
df["MSFsc"] = df["msf_sc"] / 3600
df["male"] = (df["sex"].astype(str) == "Male").astype(float)

In [ ]:
# luz dos óculos: janelas por noite
def light_features(s):
    L = raw[s]["light"][["Id", "Datetime", "MEDI", "is.implicit"]].copy()
    L.loc[L["is.implicit"], "MEDI"] = np.nan
    out = []
    for pid, dn in df[df.site == s].groupby("Id"):
        lp = L[L.Id == pid].set_index("Datetime")["MEDI"].sort_index()
        for i, r in dn.iterrows():
            B, S_ = r["bedtime"], r["sleep"]
            W = r["prev_wake"] if pd.notna(r["prev_wake"]) and 0 < (B - r["prev_wake"]).total_seconds() < 30 * 3600 else B - pd.Timedelta(hours=16)
            day = lp.loc[W:B]; exp_min = (B - W).total_seconds() / 60
            f = {"idx": i}
            if exp_min > 0 and day.notna().sum() >= 0.5 * exp_min:
                f["log_tat"] = np.log1p((day >= 250).sum())
                bright = day[day >= 1000]
                f["time1000last"] = (bright.index[-1].hour + bright.index[-1].minute / 60) if len(bright) else np.nan
            pre = lp.loc[B - pd.Timedelta(hours=3):B]
            if pre.notna().sum() >= 90: f["log_prebed"] = np.log1p(pre.mean())
            if pd.notna(S_):
                ps = lp.loc[S_ - pd.Timedelta(hours=3):S_]
                if ps.notna().sum() >= 90: f["log_presleep"] = np.log1p(ps.mean())
                lit = lp.loc[B - pd.Timedelta(hours=3):S_ + pd.Timedelta(hours=1)]
                lit = lit[lit > 1]
                if len(lit): f["lightsoff_after_bed"] = (lit.index[-1] - B).total_seconds() / 3600
            out.append(f)
    return out

lf = pd.DataFrame([f for s in SITES for f in light_features(s)]).set_index("idx")
df = df.join(lf)
df["h_tat_med"] = df.groupby("Id", group_keys=False)["log_tat"].apply(lambda s: past(s, "median"))
df["tat_dev"] = df["log_tat"] - df["h_tat_med"]
df["bed_rel_chrono"] = df["gobedTime"] - df["MSFsc"]
df["early_vs_habit"] = (-df["gobed_dev"]).clip(lower=0)
df["awake_x_chrono"] = (df["hours_awake"] - 16) * (df["bed_rel_chrono"] + 7)

# conjuntos equivalentes aos de Manchester (sem PSQI, cafeína, Fitbit e discordância)
FEAT_A = ["gobedTime", "gobed_dev", "h_n", "h_lat_med", "h_lat_mean", "h_lat_last",
          "log_tat", "tat_dev", "time1000last", "dow", "weekend_eve", "age", "male", "MSFsc"]
FEAT_S = FEAT_A + ["hours_awake", "prev_dur", "prev_eff"]
FEAT_C = FEAT_S + ["bed_rel_chrono", "early_vs_habit", "awake_x_chrono"]
FEAT_SL = FEAT_S + ["log_prebed"]                      # secundária, prospectiva
FEAT_B = FEAT_S + ["log_presleep", "lightsoff_after_bed"]  # não prospectiva
cov = df[["log_tat", "log_prebed", "log_presleep", "hours_awake", "MSFsc"]].notna().mean().round(2)
print("Cobertura das variáveis:", cov.to_dict())

hold = df[df.is_holdout].copy(); HOLD_Y = hold.pop("lat_min")
dev = df[~df.is_holdout].reset_index(drop=True)
print(f"Desenvolvimento: {len(dev)} noites, {dev.Id.nunique()} pessoas")

## 3. Modelos congelados da v3 (mesmo código e hiperparâmetros)

In [ ]:
LGB_PARAMS = dict(objective="l1", n_estimators=300, learning_rate=0.03, num_leaves=7, min_child_samples=15,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)
def pred_lgb(ms, X): return np.mean([m.predict(X) for m in ms], axis=0)
def fit_quantile(X, y):
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         QuantileRegressor(quantile=0.5, alpha=0.01, solver="highs")).fit(X, y)
LONG_THR = 30; GRID = np.arange(0, 180.5, 0.5)
def fit_twopart(tr, feats):
    long = (tr["lat_min"] > LONG_THR).to_numpy()
    clf = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                        LogisticRegression(C=0.3, max_iter=5000)).fit(tr[feats], long)
    short = tr[~long]; reg = fit_quantile(short[feats], short["lat_min"])
    return clf, reg, np.sort(short["lat_min"].to_numpy() - reg.predict(short[feats])), np.sort(tr.loc[long, "lat_min"].to_numpy())
def pred_twopart(m, te, feats):
    clf, reg, rs, lv = m; p = clf.predict_proba(te[feats])[:, 1]; ms = reg.predict(te[feats])
    Fl = np.searchsorted(lv, GRID, side="right") / len(lv)
    return np.array([GRID[np.argmax((1 - pi) * np.searchsorted(rs, GRID - mi, side="right") / len(rs) + pi * Fl >= 0.5)]
                     for pi, mi in zip(p, ms)])
def hist_pred(tr, te): return te["h_lat_med"].fillna(tr["lat_min"].median()).to_numpy()
W_GRID = np.round(np.arange(0, 1.01, 0.1), 1)
def choose_blend_weight(tr, n_inner=4, seed=SEED):
    r = np.random.default_rng(seed); ids_in = np.array(sorted(tr.Id.unique()))
    fi = tr.Id.map(dict(zip(ids_in, r.permutation(np.arange(len(ids_in)) % n_inner)))).to_numpy()
    h = np.full(len(tr), np.nan); q = np.full(len(tr), np.nan)
    for k in range(n_inner):
        a, b = tr[fi != k], tr[fi == k]
        h[fi == k] = hist_pred(a, b); q[fi == k] = fit_quantile(a[FEAT_S], a["lat_min"]).predict(b[FEAT_S])
    ok = (tr["h_n"] >= 1).to_numpy(); yy = tr["lat_min"].to_numpy()
    return float(W_GRID[int(np.argmin([np.mean(np.abs(yy[ok] - ((1 - w) * h[ok] + w * q[ok]))) for w in W_GRID]))])

def predict_block(tr, te):
    out = {"pop_median": np.full(len(te), tr["lat_min"].median()), "history": hist_pred(tr, te)}
    for name, feats in [("quantile_A", FEAT_A), ("quantile_S", FEAT_S), ("quantile_C", FEAT_C), ("quantile_S_light", FEAT_SL)]:
        out[name] = fit_quantile(tr[feats], tr["lat_min"]).predict(te[feats])
    for name, feats, kw, logt in [("lgbm_A", FEAT_A, {}, False), ("lgbm_S", FEAT_S, {}, False),
                                  ("lgbm_S_huber", FEAT_S, dict(objective="huber", alpha=15.0), False),
                                  ("lgbm_S_log", FEAT_S, {}, True), ("lgbm_C", FEAT_C, {}, False), ("lgbm_B", FEAT_B, {}, False)]:
        yt = np.log1p(tr["lat_min"]) if logt else tr["lat_min"]
        p = pred_lgb([lgb.LGBMRegressor(**{**LGB_PARAMS, **kw}, random_state=SEED + s).fit(tr[feats], yt) for s in range(5)], te[feats])
        out[name] = np.expm1(p) if logt else p
    out["twopart_S"] = pred_twopart(fit_twopart(tr, FEAT_S), te, FEAT_S)
    w = choose_blend_weight(tr); out["blend_nested"] = (1 - w) * out["history"] + w * out["quantile_S"]
    return out, w

PRIOR_W = 2.0
def personal_rate(frame, ev, p0):
    k = ev.astype(float).groupby(frame["Id"]).transform(lambda s: s.shift(1).expanding().sum()).fillna(0)
    return ((k + PRIOR_W * p0) / (frame["h_n"] + PRIOR_W)).to_numpy()
def logit(): return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                                  LogisticRegression(C=0.3, max_iter=5000))
CLF_LGB = dict(objective="binary", n_estimators=200, learning_rate=0.03, num_leaves=4, min_child_samples=20,
               subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5.0, verbose=-1)
def clf_block(tr, te, ytr, hr_te_fn):
    p0 = ytr.mean(); hr_tr = personal_rate(tr, ytr, p0); hr_te = hr_te_fn(p0)
    out = {"base_rate": np.full(len(te), p0), "hist_rate": hr_te,
           "logit_S": logit().fit(tr[FEAT_S], ytr).predict_proba(te[FEAT_S])[:, 1],
           "logit_C": logit().fit(tr[FEAT_C], ytr).predict_proba(te[FEAT_C])[:, 1]}
    Xtr, Xte = tr[FEAT_C].assign(hist_rate=hr_tr), te[FEAT_C].assign(hist_rate=hr_te)
    out["logit_C_hist"] = logit().fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    out["lgbm_clf_C"] = np.mean([lgb.LGBMClassifier(**CLF_LGB, random_state=SEED + s).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
                                 for s in range(5)], axis=0)
    return out
def mae(e): return float(np.mean(np.abs(e)))

## 4. Validação cruzada no desenvolvimento (por pessoa)

In [ ]:
rng = np.random.default_rng(SEED); ids = np.array(sorted(dev.Id.unique()))
dev["fold"] = dev.Id.map(dict(zip(ids, rng.permutation(np.arange(len(ids)) % N_FOLDS))))
yev = (dev["lat_min"] > 30).astype(int)
oof, oofc, weights = {}, {}, []
for f in range(N_FOLDS):
    tr, te = dev[dev.fold != f], dev[dev.fold == f]
    p, w = predict_block(tr, te); weights.append(w)
    pc = clf_block(tr, te, yev.loc[tr.index], lambda p0, te=te: personal_rate(te, yev.loc[te.index], p0))
    for dct, pr in [(oof, p), (oofc, pc)]:
        for k, v in pr.items(): dct.setdefault(k, pd.Series(index=dev.index, dtype=float)).loc[te.index] = v
oof, oofc = pd.DataFrame(oof), pd.DataFrame(oofc)
MAIN = dev["h_n"] >= 1
print("Pesos da combinação por dobra:", weights)

def summarize(y, P, ids_):
    rows = []
    for c in P.columns:
        e = y - P[c]
        rows.append(dict(model=c, n=len(e), MAE=mae(e), MedAE=float(e.abs().median()),
                         MAE_por_pessoa=float(e.abs().groupby(ids_).mean().mean()), RMSE=float(np.sqrt((e ** 2).mean()))))
    return pd.DataFrame(rows).set_index("model").round(2)
def paired_boot(y, P, ids_, model, ref="history", seed=SEED):
    d = (y - P[model]).abs() - (y - P[ref]).abs()
    by = d.groupby(ids_).agg(["sum", "count"]); s_, c_ = by["sum"].to_numpy(), by["count"].to_numpy()
    idx = np.random.default_rng(seed).integers(0, len(s_), size=(N_BOOT, len(s_)))
    b = s_[idx].sum(1) / c_[idx].sum(1); pp = d.groupby(ids_).mean()
    return dict(model=model, delta_MAE=round(d.mean(), 3), IC95_inf=round(np.percentile(b, 2.5), 3),
                IC95_sup=round(np.percentile(b, 97.5), 3), melhorou=int((pp < -1e-9).sum()), piorou=int((pp > 1e-9).sum()))

y_m, ids_m = dev.loc[MAIN, "lat_min"], dev.loc[MAIN, "Id"]
print(f"Latência no desenvolvimento: mediana {dev.lat_min.median():.1f} min, média {dev.lat_min.mean():.1f}, "
      f"{(dev.lat_min > 30).mean():.1%} > 30 min, {(dev.lat_min % 5 == 0).mean():.0%} múltiplos de 5")
res_dev = summarize(y_m, oof.loc[MAIN], ids_m); display(res_dev)
boot_dev = pd.DataFrame([paired_boot(y_m, oof.loc[MAIN], ids_m, m) for m in oof.columns if m != "history"]).set_index("model")
display(boot_dev)

yy = yev[MAIN]
crow = [dict(model=c, eventos=int(yy.sum()), Brier=brier_score_loss(yy, oofc.loc[MAIN, c]),
             AUC=roc_auc_score(yy, oofc.loc[MAIN, c]) if c != "base_rate" else np.nan,
             PR_AUC=average_precision_score(yy, oofc.loc[MAIN, c])) for c in oofc.columns]
res_clf_dev = pd.DataFrame(crow).set_index("model").round(4); display(res_clf_dev)

### Diagnóstico exploratório (não estava no plano): o desempenho depende do tamanho do histórico?

O reservado só tem noites com pelo menos 2 noites anteriores; o desenvolvimento inclui muitas noites com apenas 1. Esta tabela verifica se a diferença entre desenvolvimento e reservado vem daí.

In [ ]:
d = dev.loc[MAIN].copy()
d["faixa_hist"] = pd.cut(d["h_n"], [0, 1, 2, 4, 99], labels=["1 noite", "2 noites", "3–4 noites", "5+ noites"])
for m in ["history", "quantile_S", "lgbm_S"]: d["e_" + m] = (d["lat_min"] - oof.loc[MAIN, m]).abs()
display(d.groupby("faixa_hist", observed=True).agg(noites=("lat_min", "size"), MAE_history=("e_history", "mean"),
        MAE_quantile_S=("e_quantile_S", "mean"), MAE_lgbm_S=("e_lgbm_S", "mean")).round(2))
print("Maiores latências no desenvolvimento (min):", sorted(dev.lat_min)[-8:])
print("Histórico no reservado: mínimo de", int(hold.h_n.min()), "noites anteriores; mediana", int(hold.h_n.median()))

## 5. Secundária: deixando um local inteiro de fora

In [ ]:
loso = []
for s in SITES:
    tr, te = dev[dev.site != s], dev[dev.site == s]
    p, _ = predict_block(tr, te); m = (te["h_n"] >= 1).to_numpy()
    for k in ["history", "quantile_S", "blend_nested", "quantile_S_light"]:
        loso.append(dict(local=s, model=k, MAE=mae(te["lat_min"].to_numpy()[m] - p[k][m])))
loso = pd.DataFrame(loso).pivot(index="local", columns="model", values="MAE").round(2)
loso.loc["média"] = loso.mean().round(2)
display(loso)
print("Locais em que quantile_S venceu o histórico:", int((loso.drop('média').quantile_S < loso.drop('média').history).sum()), "de", len(SITES))

## 6. Reservado (comparações principais fixadas no plano)

In [ ]:
preds_h, w_h = predict_block(dev, hold.assign(lat_min=np.nan))
P = pd.DataFrame(preds_h, index=hold.index)
res_hold = summarize(HOLD_Y, P, hold.Id); display(res_hold.loc[["history", "quantile_S", "blend_nested", "quantile_S_light", "lgbm_S", "pop_median"]])
boot_hold = pd.DataFrame([paired_boot(HOLD_Y, P, hold.Id, m) for m in ["quantile_S", "blend_nested", "quantile_S_light"]]).set_index("model")
display(boot_hold)

ytr_ev = (dev["lat_min"] > 30).astype(int); kev = ytr_ev.groupby(dev.Id).sum()
pc = clf_block(dev, hold, ytr_ev, lambda p0: ((hold.Id.map(kev).fillna(0) + PRIOR_W * p0) / (hold["h_n"] + PRIOR_W)).to_numpy())
yh = (HOLD_Y > 30).astype(int)
res_clf_hold = pd.DataFrame([dict(model=c, eventos=int(yh.sum()), n=len(yh), Brier=brier_score_loss(yh, pc[c]),
                                  AUC=roc_auc_score(yh, pc[c]) if c != "base_rate" else np.nan,
                                  PR_AUC=average_precision_score(yh, pc[c])) for c in ["base_rate", "hist_rate", "logit_C_hist"]]
                            ).set_index("model").round(4)
display(res_clf_hold)
r = np.random.default_rng(SEED); idx = r.integers(0, len(yh), size=(N_BOOT, len(yh)))
for c in ["hist_rate", "logit_C_hist"]:
    d = (pc[c] - yh.values) ** 2 - (pc["base_rate"] - yh.values) ** 2
    print(f"{c}: ΔBrier vs prevalência {d.mean():+.4f}  IC95 {np.round(np.percentile(d[idx].mean(1), [2.5, 97.5]), 4)}")

                    n    MAE  MedAE  MAE_por_pessoa   RMSE
model                                                     
history           144  12.13   5.00           12.13  29.36
quantile_S        144  12.88   4.38           12.88  32.19
blend_nested      144  12.59   4.18           12.59  31.72
quantile_S_light  144  12.87   4.38           12.87  32.17
lgbm_S            144  12.05   4.17           12.05  29.91
pop_median        144  14.66   5.00           14.66  35.91

                  delta_MAE  IC95_inf  IC95_sup  melhorou  piorou
model                                                            
quantile_S            0.750    -0.806     2.357        62      82
blend_nested          0.462    -1.040     1.977        62      82
quantile_S_light      0.737    -0.814     2.331        62      82

              eventos    n   Brier     AUC  PR_AUC
model                                             
base_rate          18  144  0.1094     NaN  0.1250
hist_rate          18  144  0.0781  0.8223  0.5130
logit_C_hist       18  144  0.0824  0.8250  0.5461

## 7. Comparação entre coortes

In [ ]:
manchester = {  # resultados da v3 (Manchester), já publicados no notebook anterior
    "dev_MAE":  {"history": 9.66, "quantile_S": 9.85, "blend_nested": 9.86},
    "hold_MAE": {"history": 11.94, "quantile_S": 11.51, "blend_nested": 11.57},
    "hold_clf": {"hist_rate_Brier": 0.1302, "hist_rate_AUC": 0.7623, "base_Brier": 0.1361},
}
comp = pd.DataFrame({
    ("Manchester", "desenvolvimento"): manchester["dev_MAE"],
    ("Manchester", "reservado"): manchester["hold_MAE"],
    ("MeLiDos", "desenvolvimento"): res_dev.loc[["history", "quantile_S", "blend_nested"], "MAE"].to_dict(),
    ("MeLiDos", "reservado"): res_hold.loc[["history", "quantile_S", "blend_nested"], "MAE"].to_dict(),
})
print("Erro absoluto médio (min)"); display(comp)
print("Classificação no reservado (> 30 min):")
display(pd.DataFrame({"Manchester": [manchester["hold_clf"]["base_Brier"], manchester["hold_clf"]["hist_rate_Brier"], manchester["hold_clf"]["hist_rate_AUC"]],
                      "MeLiDos": [res_clf_hold.loc["base_rate", "Brier"], res_clf_hold.loc["hist_rate", "Brier"], res_clf_hold.loc["hist_rate", "AUC"]]},
                     index=["Brier prevalência", "Brier hist_rate", "AUC hist_rate"]).round(4))

                  Manchester                   MeLiDos          
             desenvolvimento reservado desenvolvimento reservado
history                 9.66     11.94           14.89     12.13
quantile_S              9.85     11.51           12.94     12.88
blend_nested            9.86     11.57           13.00     12.59

                   Manchester  MeLiDos
Brier prevalência      0.1361   0.1094
Brier hist_rate        0.1302   0.0781
AUC hist_rate          0.7623   0.8223

In [ ]:
summary = dict(sites=list(SITES), excluded={"UCR": "já analisado no teste lunar"}, holdout_keys_sha256=HOLDOUT_SHA,
               n_dev=len(dev), n_hold=len(hold), blend_weights=weights, blend_weight_holdout=w_h,
               dev=res_dev.reset_index().to_dict("records"), holdout=res_hold.reset_index().to_dict("records"),
               holdout_boot=boot_hold.reset_index().to_dict("records"), clf_holdout=res_clf_hold.reset_index().to_dict("records"),
               loso=loso.reset_index().to_dict("records"))
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("Resumo salvo (agregado; sem dados de participantes).")

In [ ]:
# Exportação harmonizada para o cruzamento entre coortes (arquivo local: contém dados de participantes, não publicar)
COMMON = ["gobedTime", "gobed_dev", "h_n", "h_lat_med", "h_lat_mean", "h_lat_last", "log_tat", "tat_dev",
          "time1000last", "dow", "weekend_eve", "age", "male", "MSFsc", "hours_awake", "prev_dur", "prev_eff"]
xp = df[["Id", "evening", "is_holdout", "site", "lat_min"] + COMMON].copy()
xp["cohort"] = "MeLiDos"
xp.to_csv(OUT_DIR / "harmonized_melidos.csv", index=False)
print("Exportado:", xp.shape)

In [ ]:
d_ = dev.loc[MAIN].copy()
d_["faixa"] = pd.cut(d_["h_n"], [0, 1, 2, 4, 99], labels=["1", "2", "3-4", "5+"])
for m in ["history", "quantile_S", "lgbm_S"]: d_["e_" + m] = (d_["lat_min"] - oof.loc[MAIN, m]).abs()
r_ = np.random.default_rng(SEED); idx_ = r_.integers(0, len(yh), size=(N_BOOT, len(yh)))
ART["mel"] = dict(
    dev=res_dev.copy(), boot_dev=boot_dev.copy(), clf_dev=res_clf_dev.copy(), hold=res_hold.copy(),
    boot_hold=boot_hold.copy(), clf_hold=res_clf_hold.copy(), loso=loso.copy(),
    diag=d_.groupby("faixa", observed=True)[["e_history", "e_quantile_S", "e_lgbm_S"]].mean(),
    hold_dbrier=(lambda d: (float(d.mean()), float(np.percentile(d[idx_].mean(1), 2.5)), float(np.percentile(d[idx_].mean(1), 97.5))))(
        (pc["hist_rate"] - yh.values) ** 2 - (pc["base_rate"] - yh.values) ** 2),
    n_hold=len(hold), ev_hold=int(yh.sum()), ev_dev=int((dev.loc[MAIN, "lat_min"] > 30).sum()), n_main=int(MAIN.sum()),
    lat=(float(dev.lat_min.median()), float(dev.lat_min.mean()), float((dev.lat_min % 5 == 0).mean()), float(dev.lat_min.max())),
    hold_hn=(int(hold.h_n.min()), float(hold.h_n.median())),
)
print("MeLiDos guardado.")

# Parte C — Cruzamento entre coortes

## SpaceSleepTwin — Cruzamento Manchester × MeLiDos

**Pré-requisito:** rodar antes `manchester_competition_v3_holdout.ipynb` e `melidos_replication.ipynb`, que geram os arquivos harmonizados locais (`harmonized_*.csv`, com dados de participantes: não publicar).

**Status: exploratório.** Os dois reservados já foram abertos, então nenhuma análise aqui tem valor de confirmação. Para limitar escolhas feitas olhando o resultado, os modelos e hiperparâmetros são os mesmos da v3, sem ajuste, e todas as análises abaixo foram definidas antes de rodar.

**Três perguntas:**
1. **Transferência:** um modelo treinado só numa coorte prevê a outra? Mede se o que se aprende generaliza entre países, sensores e protocolos.
2. **Agrupamento:** treinar com as duas coortes juntas ajuda cada uma, comparado a treinar só dentro dela?
3. **Diferença entre coortes:** a mesma variável tem o mesmo efeito nas duas?

**Variáveis comuns** (17): horário de deitar e desvio do habitual, histórico pessoal, luz diurna (`log_tat`, `tat_dev`, `time1000last`), calendário, idade, sexo, cronotipo, horas acordada, duração e eficiência do sono anterior.

**Diferenças que limitam a comparação:**
| | Manchester | MeLiDos |
|---|---|---|
| Latência | deitar → dormir | início da tentativa de dormir → dormir |
| Luz | sensor de pulso (Spectrawear) | óculos, nível dos olhos |
| Idade | faixa etária (início da faixa) | idade exata |
| Locais | Reino Unido | 8 locais em 6 países (Alemanha, Espanha, Gana, Holanda, Suécia, Turquia; sem UCR) |

In [ ]:
import pathlib, numpy as np, pandas as pd, warnings
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import QuantileRegressor, LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import lightgbm as lgb
warnings.filterwarnings("ignore"); pd.set_option("display.width", 170)
SEED, N_FOLDS, N_BOOT = 20261011, 5, 2000
M = pd.read_csv("results_manchester_competition/harmonized_manchester.csv", parse_dates=["evening"])
L = pd.read_csv("results_melidos_replication/harmonized_melidos.csv", parse_dates=["evening"])
COMMON = ["gobedTime", "gobed_dev", "h_n", "h_lat_med", "h_lat_mean", "h_lat_last", "log_tat", "tat_dev",
          "time1000last", "dow", "weekend_eve", "age", "male", "MSFsc", "hours_awake", "prev_dur", "prev_eff"]
X = pd.concat([M, L], ignore_index=True)
X["cohort_mel"] = (X["cohort"] == "MeLiDos").astype(int)
print(X.groupby("cohort").agg(noites=("lat_min", "size"), pessoas=("Id", "nunique"),
                              mediana_lat=("lat_min", "median"), p_maior_30=("lat_min", lambda s: (s > 30).mean())).round(3))

## Distribuição das variáveis comuns por coorte (mudança de domínio)

In [ ]:
display(X.groupby("cohort")[COMMON].median().T.round(2))

In [ ]:
LGB = dict(objective="l1", n_estimators=300, learning_rate=0.03, num_leaves=7, min_child_samples=15,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, verbose=-1)
def fit_q(Xt, y): return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                                       QuantileRegressor(quantile=0.5, alpha=0.01, solver="highs")).fit(Xt, y)
def logit(): return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                                  LogisticRegression(C=0.3, max_iter=5000))
PRIOR_W = 2.0
def hist_rate(frame, p0):
    ev = (frame["lat_min"] > 30).astype(float)
    k = ev.groupby(frame["Id"]).transform(lambda s: s.shift(1).expanding().sum()).fillna(0)
    return ((k + PRIOR_W * p0) / (frame["h_n"] + PRIOR_W)).to_numpy()

def run(tr, te, feats):
    out = {"history": te["h_lat_med"].fillna(tr["lat_min"].median()).to_numpy()}
    out["quantile"] = fit_q(tr[feats], tr["lat_min"]).predict(te[feats])
    out["lgbm"] = np.mean([lgb.LGBMRegressor(**LGB, random_state=SEED + s).fit(tr[feats], tr["lat_min"]).predict(te[feats]) for s in range(5)], axis=0)
    ytr = (tr["lat_min"] > 30).astype(int); p0 = ytr.mean()
    hr_tr, hr_te = hist_rate(tr, p0), hist_rate(te, p0)
    out["p_base"] = np.full(len(te), p0); out["p_hist"] = hr_te
    out["p_logit_hist"] = logit().fit(tr[feats].assign(hr=hr_tr), ytr).predict_proba(te[feats].assign(hr=hr_te))[:, 1]
    return out

def evaluate(te, out, label):
    m = (te["h_n"] >= 1).to_numpy(); y = te["lat_min"].to_numpy()[m]; ids = te["Id"].to_numpy()[m]
    yb = (y > 30).astype(int); r = np.random.default_rng(SEED)
    uid = np.unique(ids); idx = r.integers(0, len(uid), size=(N_BOOT, len(uid)))
    row = dict(cenario=label, noites=int(m.sum()))
    for k in ["history", "quantile", "lgbm"]:
        row[f"MAE_{k}"] = np.mean(np.abs(y - out[k][m]))
    for k in ["quantile", "lgbm"]:
        d = pd.Series(np.abs(y - out[k][m]) - np.abs(y - out["history"][m])).groupby(ids).agg(["sum", "count"]).loc[uid]
        b = d["sum"].to_numpy()[idx].sum(1) / d["count"].to_numpy()[idx].sum(1)
        row[f"Δ{k}_IC95"] = f"{(d['sum'].sum()/d['count'].sum()):+.2f} [{np.percentile(b,2.5):+.2f}; {np.percentile(b,97.5):+.2f}]"
    for k in ["p_base", "p_hist", "p_logit_hist"]:
        row[f"Brier_{k}"] = brier_score_loss(yb, out[k][m])
    row["AUC_hist"] = roc_auc_score(yb, out["p_hist"][m]); row["AUC_logit_hist"] = roc_auc_score(yb, out["p_logit_hist"][m])
    return row

def cv_within(frame, feats, label, extra_frame=None):
    # GroupKFold por pessoa dentro de `frame`; se extra_frame for dado, ele entra sempre no treino (agrupamento)
    r = np.random.default_rng(SEED); ids = np.array(sorted(frame.Id.unique()))
    fold = frame.Id.map(dict(zip(ids, r.permutation(np.arange(len(ids)) % N_FOLDS)))).to_numpy()
    preds = {}
    for f in range(N_FOLDS):
        tr = frame[fold != f]; te = frame[fold == f]
        if extra_frame is not None: tr = pd.concat([tr, extra_frame], ignore_index=True)
        o = run(tr, te, feats)
        for k, v in o.items(): preds.setdefault(k, np.full(len(frame), np.nan))[fold == f] = v
    return evaluate(frame, preds, label)

## 1–2. Dentro da coorte, transferência e agrupamento

- **Dentro:** validação por pessoa só com a própria coorte (referência).
- **Transferência:** treina com **todas** as noites da outra coorte e prevê esta. O histórico pessoal continua vindo das noites anteriores da própria pessoa.
- **Agrupado:** validação por pessoa nesta coorte, com a outra coorte inteira somada ao treino e um indicador de coorte.

In [ ]:
rows = []
for name, own, other in [("Manchester", M, L), ("MeLiDos", L, M)]:
    rows.append(cv_within(own, COMMON, f"{name} · dentro"))
    rows.append(evaluate(own, run(other, own, COMMON), f"{name} · transferência ({'MeLiDos' if name=='Manchester' else 'Manchester'} → {name})"))
    own2 = own.assign(cohort_mel=int(name == "MeLiDos")); oth2 = other.assign(cohort_mel=int(name != "MeLiDos"))
    rows.append(cv_within(own2, COMMON + ["cohort_mel"], f"{name} · agrupado", extra_frame=oth2))
res = pd.DataFrame(rows).set_index("cenario")
reg_cols = ["noites", "MAE_history", "MAE_quantile", "MAE_lgbm", "Δquantile_IC95", "Δlgbm_IC95"]
clf_cols = ["Brier_p_base", "Brier_p_hist", "Brier_p_logit_hist", "AUC_hist", "AUC_logit_hist"]
print("Regressão (min). Δ = modelo − histórico; negativo = melhor; IC95 por bootstrap de pessoas")
display(res[reg_cols].round(2))
print("Classificação (latência > 30 min)")
display(res[clf_cols].round(4))

## 3. A mesma variável tem o mesmo efeito nas duas coortes?

Regressão quantílica na mediana com todas as noites das duas coortes, variáveis padronizadas, e termos de interação `variável × MeLiDos` para as principais. Coeficientes descritivos: sem intervalos, exploratórios.

In [ ]:
KEY = ["h_lat_med", "log_tat", "time1000last", "hours_awake", "gobed_dev", "MSFsc"]
Z = X[X.h_n >= 1].copy()
imp = SimpleImputer(strategy="median").fit(Z[COMMON]); Zc = pd.DataFrame(imp.transform(Z[COMMON]), columns=COMMON, index=Z.index)
Zc = (Zc - Zc.mean()) / Zc.std()
for k in KEY: Zc[k + "×MeLiDos"] = Zc[k] * Z["cohort_mel"]
Zc["MeLiDos"] = Z["cohort_mel"]
qr = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs").fit(Zc, Z["lat_min"])
coef = pd.Series(qr.coef_, index=Zc.columns)
tab = pd.DataFrame({"Manchester": coef[KEY].values, "MeLiDos": (coef[KEY].values + coef[[k + "×MeLiDos" for k in KEY]].values)}, index=KEY)
print("Efeito na mediana da latência (min por 1 desvio padrão da variável)")
display(tab.round(2))

              Manchester  MeLiDos
h_lat_med          14.93     7.10
log_tat            -2.81    -1.07
time1000last        0.46    -0.37
hours_awake        -0.24    -1.18
gobed_dev          -0.79    -0.11
MSFsc               0.46     0.35

## Como ler

- Se a **transferência** ficar perto do **dentro**, o que os modelos aprendem é geral; se ficar muito pior, depende da coorte (sensor, protocolo, população).
- Se o **agrupado** for melhor que o **dentro**, mais dados de outra coorte ajudam.
- O histórico pessoal não depende de treino, então muda pouco entre cenários: ele é a régua.

In [ ]:
ART["cross"] = dict(res=res.copy(), coef=tab.copy())
print("Cruzamento guardado.")

# Parte D — Conferência dos números do artigo

Cada valor citado no artigo é comparado com o recalculado acima (tolerância de meia unidade na última casa exibida). Qualquer `DIFERENTE` indica que o texto do artigo precisa ser atualizado.

In [ ]:
A, Me, X_ = ART["man"], ART["mel"], ART["cross"]
def ci(t): return t
checks = [
 # (descrição, valor no artigo, recalculado, casas)
 ("Man dev MAE histórico", 9.66, A["dev"].loc["history","MAE"], 2),
 ("Man dev MAE quantílica S", 9.85, A["dev"].loc["quantile_S","MAE"], 2),
 ("Man dev MAE combinação", 9.86, A["dev"].loc["blend_nested","MAE"], 2),
 ("Man dev MAE LightGBM S", 10.43, A["dev"].loc["lgbm_S","MAE"], 2),
 ("Man dev ΔMAE quantílica S", 0.185, A["boot_dev"].loc["quantile_S","delta_MAE"], 3),
 ("Man dev IC inf", -1.19, A["boot_dev"].loc["quantile_S","IC95_inf"], 2),
 ("Man dev IC sup", 1.99, A["boot_dev"].loc["quantile_S","IC95_sup"], 2),
 ("Man reservado MAE histórico", 11.94, A["hold_mae"]["history"], 2),
 ("Man reservado MAE quantílica S", 11.51, A["hold_mae"]["quantile_S"], 2),
 ("Man reservado MAE combinação", 11.57, A["hold_mae"]["blend_nested"], 2),
 ("Man reservado ΔMAE quantílica S", -0.43, A["hold_boot"]["quantile_S"][0], 2),
 ("Man reservado IC inf", -1.98, A["hold_boot"]["quantile_S"][1], 2),
 ("Man reservado IC sup", 1.00, A["hold_boot"]["quantile_S"][2], 2),
 ("Man reservado Brier prevalência", 0.136, A["hold_base_brier"], 3),
 ("Man reservado Brier hist_rate", 0.130, A["hold_clf"]["hist_rate"][0], 3),
 ("Man reservado AUC hist_rate", 0.76, A["hold_clf"]["hist_rate"][1], 2),
 ("Man reservado Brier logística", 0.143, A["hold_clf"]["logit_C_hist"][0], 3),
 ("Man reservado AUC logística", 0.65, A["hold_clf"]["logit_C_hist"][1], 2),
 ("Man reservado ΔBrier", -0.006, A["hold_dbrier"][0], 3),
 ("Man reservado ΔBrier IC inf", -0.043, A["hold_dbrier"][1], 3),
 ("Man reservado ΔBrier IC sup", 0.032, A["hold_dbrier"][2], 3),
 ("Man dev Brier hist_rate", 0.091, A["clf_dev"].loc["hist_rate","Brier"], 3),
 ("Man dev AUC hist_rate", 0.77, A["clf_dev"].loc["hist_rate","AUC"], 2),
 ("Man sensibilidade Brier hist_rate", 0.131, A["clf_sens"].loc["hist_rate","Brier"], 3),
 ("Man sensibilidade AUC hist_rate", 0.81, A["clf_sens"].loc["hist_rate","AUC"], 2),
 ("Man eventos dev", 37, A["ev_dev"], 0), ("Man eventos reservado", 13, A["ev_hold"], 0),
 ("Mel dev MAE histórico", 14.89, Me["dev"].loc["history","MAE"], 2),
 ("Mel dev MAE quantílica S", 12.94, Me["dev"].loc["quantile_S","MAE"], 2),
 ("Mel dev MAE combinação", 13.00, Me["dev"].loc["blend_nested","MAE"], 2),
 ("Mel dev MAE LightGBM S", 12.70, Me["dev"].loc["lgbm_S","MAE"], 2),
 ("Mel dev MAE com luz pré-deitar", 12.96, Me["dev"].loc["quantile_S_light","MAE"], 2),
 ("Mel dev ΔMAE quantílica S", -1.95, Me["boot_dev"].loc["quantile_S","delta_MAE"], 2),
 ("Mel dev IC inf", -3.74, Me["boot_dev"].loc["quantile_S","IC95_inf"], 2),
 ("Mel dev IC sup", -0.47, Me["boot_dev"].loc["quantile_S","IC95_sup"], 2),
 ("Mel reservado MAE histórico", 12.13, Me["hold"].loc["history","MAE"], 2),
 ("Mel reservado MAE quantílica S", 12.88, Me["hold"].loc["quantile_S","MAE"], 2),
 ("Mel reservado MAE combinação", 12.59, Me["hold"].loc["blend_nested","MAE"], 2),
 ("Mel reservado MAE LightGBM S", 12.05, Me["hold"].loc["lgbm_S","MAE"], 2),
 ("Mel reservado MAE com luz pré-deitar", 12.87, Me["hold"].loc["quantile_S_light","MAE"], 2),
 ("Mel reservado ΔMAE quantílica S", 0.75, Me["boot_hold"].loc["quantile_S","delta_MAE"], 2),
 ("Mel reservado IC inf", -0.81, Me["boot_hold"].loc["quantile_S","IC95_inf"], 2),
 ("Mel reservado IC sup", 2.36, Me["boot_hold"].loc["quantile_S","IC95_sup"], 2),
 ("Mel reservado Brier prevalência", 0.109, Me["clf_hold"].loc["base_rate","Brier"], 3),
 ("Mel reservado Brier hist_rate", 0.078, Me["clf_hold"].loc["hist_rate","Brier"], 3),
 ("Mel reservado AUC hist_rate", 0.82, Me["clf_hold"].loc["hist_rate","AUC"], 2),
 ("Mel reservado PR-AUC hist_rate", 0.51, Me["clf_hold"].loc["hist_rate","PR_AUC"], 2),
 ("Mel reservado Brier logística", 0.082, Me["clf_hold"].loc["logit_C_hist","Brier"], 3),
 ("Mel reservado AUC logística", 0.825, Me["clf_hold"].loc["logit_C_hist","AUC"], 3),
 ("Mel reservado ΔBrier", -0.031, Me["hold_dbrier"][0], 3),
 ("Mel reservado ΔBrier IC inf", -0.057, Me["hold_dbrier"][1], 3),
 ("Mel reservado ΔBrier IC sup", -0.006, Me["hold_dbrier"][2], 3),
 ("Mel dev Brier hist_rate", 0.087, Me["clf_dev"].loc["hist_rate","Brier"], 3),
 ("Mel dev AUC hist_rate", 0.78, Me["clf_dev"].loc["hist_rate","AUC"], 2),
 ("Mel eventos dev", 87, Me["ev_dev"], 0), ("Mel eventos reservado", 18, Me["ev_hold"], 0),
 ("Mel diag 1 noite histórico", 19.63, Me["diag"].loc["1","e_history"], 2),
 ("Mel diag 5+ histórico", 11.00, Me["diag"].loc["5+","e_history"], 2),
 ("Mel diag 5+ quantílica", 10.33, Me["diag"].loc["5+","e_quantile_S"], 2),
 ("Mel diag 5+ LightGBM", 9.75, Me["diag"].loc["5+","e_lgbm_S"], 2),
 ("Mel locais em que quantílica venceu", 6, int((Me["loso"].drop("média").quantile_S < Me["loso"].drop("média").history).sum()), 0),
 ("Cruz Man histórico", 10.18, X_["res"].loc["Manchester · dentro","MAE_history"], 2),
 ("Cruz Man LightGBM dentro", 10.49, X_["res"].loc["Manchester · dentro","MAE_lgbm"], 2),
 ("Cruz Man LightGBM transferido", 10.02, X_["res"].loc["Manchester · transferência (MeLiDos → Manchester)","MAE_lgbm"], 2),
 ("Cruz Man LightGBM agrupado", 9.84, X_["res"].loc["Manchester · agrupado","MAE_lgbm"], 2),
 ("Cruz Mel histórico", 14.42, X_["res"].loc["MeLiDos · dentro","MAE_history"], 2),
 ("Cruz Mel LightGBM dentro", 12.68, X_["res"].loc["MeLiDos · dentro","MAE_lgbm"], 2),
 ("Cruz Mel LightGBM transferido", 12.87, X_["res"].loc["MeLiDos · transferência (Manchester → MeLiDos)","MAE_lgbm"], 2),
 ("Cruz Mel LightGBM agrupado", 12.46, X_["res"].loc["MeLiDos · agrupado","MAE_lgbm"], 2),
 ("Cruz luz diurna Manchester", -2.8, X_["coef"].loc["log_tat","Manchester"], 1),
 ("Cruz luz diurna MeLiDos", -1.1, X_["coef"].loc["log_tat","MeLiDos"], 1),
 ("Cruz horas acordada MeLiDos", -1.2, X_["coef"].loc["hours_awake","MeLiDos"], 1),
 ("Cruz histórico Manchester", 14.9, X_["coef"].loc["h_lat_med","Manchester"], 1),
 ("Cruz histórico MeLiDos", 7.1, X_["coef"].loc["h_lat_med","MeLiDos"], 1),
]
rows = []
for desc, art, calc, nd in checks:
    ok = abs(round(float(calc), nd) - art) <= 0.5 * 10 ** (-nd) + 1e-9
    rows.append(dict(item=desc, artigo=art, recalculado=round(float(calc), nd + 1), status="OK" if ok else "DIFERENTE"))
chk = pd.DataFrame(rows)
pd.set_option("display.max_rows", 200)
display(chk)
print(f"{(chk.status == 'OK').sum()} de {len(chk)} números conferem.")

## Observações

- O bootstrap é descritivo e não corrige múltiplas comparações.
- O reservado de Manchester foi aberto uma vez, com os modelos fixados na v3, sem commit de lacre anterior.
- A Parte C usa todas as noites e as 17 variáveis comuns; seus números não se comparam às Partes A e B.